# LeetCode 1583 — Count Unhappy Friends bằng Python

Link đề: https://leetcode.com/problems/count-unhappy-friends/ · Độ khó: **Medium** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def unhappyFriends(self, n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc chính xác định nghĩa "không vui" (unhappy) gồm **hai điều kiện đồng thời** và dịch nó thành code theo đúng nghĩa đen (brute force).
2. Xây **bảng thứ hạng** `rank[x][y]` = vị trí của `y` trong danh sách ưu tiên của `x`, để so sánh "ai thích ai hơn" trong `O(1)` thay vì `list.index` `O(n)`.
3. Dùng mảng `partner[x]` để tra bạn cặp của mỗi người trong `O(1)`.
4. Tối ưu thực tế: chỉ duyệt những người `x` **thích hơn** bạn cặp của mình, và **dừng sớm** khi tìm thấy một bằng chứng.
5. Nhận ra tính **đối xứng**: nếu `(x, u)` là một "cặp chặn" thì cả `x` lẫn `u` đều không vui — liên hệ với khái niệm **ghép cặp ổn định** (stable matching).

## 2. Đề bài & ví dụ

Có `n` người bạn (`n` chẵn), đánh số `0..n-1`. `preferences[i]` là danh sách những người bạn còn lại **sắp theo mức độ yêu thích giảm dần** (người đứng trước được thích hơn).

Tất cả được chia thành các cặp theo `pairs`, trong đó `pairs[i] = [xᵢ, yᵢ]` nghĩa là `xᵢ` ghép với `yᵢ` (và ngược lại).

Một người `x` **không vui** nếu `x` ghép với `y` và tồn tại người `u` ghép với `v` sao cho:
- `x` thích `u` hơn `y`, **và**
- `u` thích `x` hơn `v`.

Trả về **số người không vui**.

**Ví dụ 1:** `n = 4`, `preferences = [[1,2,3],[3,2,0],[3,1,0],[1,2,0]]`, `pairs = [[0,1],[2,3]]` → `2`.
Người 1 không vui (ghép với 0, thích 3 hơn 0; 3 thích 1 hơn 2). Người 3 không vui (ghép với 2, thích 1 hơn 2; 1 thích 3 hơn 0). Người 0 và 2 vui.

**Ví dụ 2:** `n = 2`, `preferences = [[1],[0]]`, `pairs = [[1,0]]` → `0`.

**Ví dụ 3:** `n = 4`, `preferences = [[1,3,2],[2,3,0],[1,3,0],[0,2,1]]`, `pairs = [[1,3],[0,2]]` → `4`.

**Ràng buộc:**
- `2 ≤ n ≤ 500`, `n` chẵn.
- `preferences.length == n`, `preferences[i].length == n - 1`, `preferences[i]` không chứa `i`, các giá trị phân biệt trong `[0, n-1]`.
- `pairs.length == n/2`, mỗi người xuất hiện trong đúng một cặp, `xᵢ ≠ yᵢ`.

In [1]:
import random
import time
from typing import List

examples = [
    (4, [[1, 2, 3], [3, 2, 0], [3, 1, 0], [1, 2, 0]], [[0, 1], [2, 3]], 2),
    (2, [[1], [0]], [[1, 0]], 0),
    (4, [[1, 3, 2], [2, 3, 0], [1, 3, 0], [0, 2, 1]], [[1, 3], [0, 2]], 4),
]
for n, prefs, pairs, want in examples:
    print(f"n={n}, preferences={prefs}, pairs={pairs} -> mong đợi {want}")

n=4, preferences=[[1, 2, 3], [3, 2, 0], [3, 1, 0], [1, 2, 0]], pairs=[[0, 1], [2, 3]] -> mong đợi 2
n=2, preferences=[[1], [0]], pairs=[[1, 0]] -> mong đợi 0
n=4, preferences=[[1, 3, 2], [2, 3, 0], [1, 3, 0], [0, 2, 1]], pairs=[[1, 3], [0, 2]] -> mong đợi 4


## 3. Kiến thức nền

### 3.1 `list.index` — tìm vị trí của một phần tử

`preferences[x].index(u)` trả về vị trí của `u` trong danh sách ưu tiên của `x` (càng nhỏ càng được thích). Mỗi lần gọi tốn `O(n)` vì phải dò tuyến tính.

In [2]:
prefs0 = [1, 2, 3]
for u in [1, 2, 3]:
    print(f"vị trí của {u} trong danh sách của người 0:", prefs0.index(u))
print("0 thích 3 hơn 1?", prefs0.index(3) < prefs0.index(1))

vị trí của 1 trong danh sách của người 0: 0
vị trí của 2 trong danh sách của người 0: 1
vị trí của 3 trong danh sách của người 0: 2
0 thích 3 hơn 1? False


### 3.2 Bảng thứ hạng (rank matrix) — "đảo ngược" danh sách ưu tiên

`rank[x][u] = i` nếu `preferences[x][i] == u`. Xây một lần `O(n²)`, sau đó mọi phép so sánh "x thích u hơn y?" chỉ là `rank[x][u] < rank[x][y]` — `O(1)`.

In [3]:
prefs = [[1, 2, 3], [3, 2, 0], [3, 1, 0], [1, 2, 0]]
n = 4
rank = [[0] * n for _ in range(n)]
for x in range(n):
    for i, u in enumerate(prefs[x]):
        rank[x][u] = i
for x in range(n):
    print(f"rank[{x}] = {rank[x]}  (rank[x][x] không dùng)")

rank[0] = [0, 0, 1, 2]  (rank[x][x] không dùng)
rank[1] = [2, 0, 1, 0]  (rank[x][x] không dùng)
rank[2] = [2, 1, 0, 0]  (rank[x][x] không dùng)
rank[3] = [2, 0, 1, 0]  (rank[x][x] không dùng)


### 3.3 Mảng `partner` từ danh sách cặp

`partner[x] = y` và `partner[y] = x` cho mỗi cặp `[x, y]`.

In [4]:
pairs = [[0, 1], [2, 3]]
partner = [0] * 4
for x, y in pairs:
    partner[x], partner[y] = y, x
print("partner =", partner)

partner = [1, 0, 3, 2]


### 3.4 `any(...)` và dừng sớm

`any(điều_kiện for ...)` trả `True` ngay khi gặp phần tử đầu tiên thoả, không duyệt hết. Phù hợp với "**tồn tại** người `u` sao cho …".

In [5]:
calls = []
def check(v):
    calls.append(v)
    return v > 2
print("any:", any(check(v) for v in [1, 3, 5, 7]), "| đã kiểm tra:", calls)

any: True | đã kiểm tra: [1, 3]


### 3.5 Cắt lát danh sách (slicing) để lấy "những người được thích hơn"

`preferences[x][:rank[x][y]]` là danh sách những người mà `x` thích **hơn** bạn cặp `y`.

In [6]:
x, y = 1, 0
better = prefs[x][:rank[x][y]]
print(f"người {x} ghép với {y}; những người {x} thích hơn {y}:", better)

người 1 ghép với 0; những người 1 thích hơn 0: [3, 2]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `preferences = [[1,2,3],[3,2,0],[3,1,0],[1,2,0]]`, cặp `(0,1)`, `(2,3)`.

| x | bạn cặp y | những người x thích hơn y | với mỗi u: u ghép với v, u thích x hơn v? | x không vui? |
|---|---|---|---|---|
| 0 | 1 | `[]` (1 đứng đầu) | — | không |
| 1 | 0 | `[3, 2]` | u=3, v=2: danh sách của 3 là `[1,2,0]`, 1 trước 2 ✔ | **có** |
| 2 | 3 | `[]` (3 đứng đầu) | — | không |
| 3 | 2 | `[1]` | u=1, v=0: danh sách của 1 là `[3,2,0]`, 3 trước 0 ✔ | **có** |

Kết quả `2`. Để ý: cặp `(1, 3)` "chặn" — cả hai đều thích nhau hơn bạn cặp hiện tại ⇒ **cả hai** đều không vui.

In [7]:
prefs = [[1, 2, 3], [3, 2, 0], [3, 1, 0], [1, 2, 0]]
partner = [1, 0, 3, 2]
for x in range(4):
    y = partner[x]
    better = prefs[x][:prefs[x].index(y)]
    witnesses = [u for u in better if prefs[u].index(x) < prefs[u].index(partner[u])]
    print(f"x={x}, y={y}, thích hơn y: {better}, bằng chứng u: {witnesses} -> {'KHÔNG VUI' if witnesses else 'vui'}")

x=0, y=1, thích hơn y: [], bằng chứng u: [] -> vui
x=1, y=0, thích hơn y: [3, 2], bằng chứng u: [3] -> KHÔNG VUI
x=2, y=3, thích hơn y: [], bằng chứng u: [] -> vui
x=3, y=2, thích hơn y: [1], bằng chứng u: [1] -> KHÔNG VUI


## 5. Cách 1 — Brute Force / baseline: dịch nguyên văn định nghĩa

Với mỗi `x`: lấy `y = partner(x)` bằng cách **dò danh sách `pairs`**; với mỗi `u ∉ {x, y}`, lấy `v = partner(u)`; kiểm tra hai điều kiện bằng `list.index`.

In [8]:
def brute_force(n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
    def find_partner(p):
        for a, b in pairs:
            if a == p:
                return b
            if b == p:
                return a

    unhappy = 0
    for x in range(n):
        y = find_partner(x)
        for u in range(n):
            if u == x or u == y:
                continue
            v = find_partner(u)
            if preferences[x].index(u) < preferences[x].index(y) and \
               preferences[u].index(x) < preferences[u].index(v):
                unhappy += 1
                break                     # x đã không vui, không cần tìm thêm u
    return unhappy

for n, prefs, pairs, want in examples:
    print(brute_force(n, prefs, pairs), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0
4 | mong đợi 4


**Độ phức tạp:** với mỗi cặp `(x, u)` (`n²` cặp): tìm bạn cặp `O(n)` + bốn lần `index` `O(n)` ⇒ `O(n³)`. Với `n = 500`: ~`10⁸`–`10⁹` phép — quá chậm trong Python.

**Nút thắt:** `find_partner` và `index` đều là **tra cứu tuyến tính lặp đi lặp lại**. Tiền xử lý thành mảng `partner` và bảng `rank` sẽ biến chúng thành `O(1)`.

In [9]:
def random_instance(n, rng):
    prefs = []
    for i in range(n):
        others = [j for j in range(n) if j != i]
        rng.shuffle(others)
        prefs.append(others)
    people = list(range(n))
    rng.shuffle(people)
    pairs = [[people[k], people[k + 1]] for k in range(0, n, 2)]
    return prefs, pairs

rng = random.Random(1)
prefs_t, pairs_t = random_instance(120, rng)
t0 = time.perf_counter()
r = brute_force(120, prefs_t, pairs_t)
print(f"n=120: {r} người không vui, mất {time.perf_counter() - t0:.3f}s")

n=120: 119 người không vui, mất 0.002s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tiền xử lý tra cứu:**
- `partner[x]` — `O(n)` để xây, `O(1)` để tra.
- `rank[x][u]` — `O(n²)` để xây, `O(1)` để tra. Khi đó "x thích u hơn y" ⇔ `rank[x][u] < rank[x][y]`.

**Insight 2 — chỉ cần xét những người `x` thích hơn `y`:** điều kiện 1 nói `u` phải đứng trước `y` trong `preferences[x]` ⇒ duyệt `u in preferences[x][:rank[x][y]]`. Nếu `y` đứng đầu danh sách của `x` thì `x` chắc chắn vui.

**Insight 3 — dừng sớm:** chỉ cần **một** `u` thoả là `x` không vui ⇒ `break` (hoặc `any`).

**Insight 4 — đối xứng:** `(x, u)` thoả hai điều kiện ⇔ `x` thích `u` hơn bạn cặp và `u` thích `x` hơn bạn cặp — điều kiện đối xứng giữa `x` và `u`. Vậy `x` không vui ⇔ `x` thuộc một **cặp chặn** (blocking pair) nào đó. Có thể đánh dấu cả hai vào một `set`.

**Quy tắc:** *nếu* `rank[x][u] < rank[x][partner[x]]` *và* `rank[u][x] < rank[u][partner[u]]` *thì* `x` (và `u`) không vui.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bảng `rank` + mảng `partner`, duyệt mọi `u`

**Đoạn 1:** xây `rank` (`n × n`) và `partner`.

**Đoạn 2:** với mỗi `x`, duyệt mọi `u ≠ x` và kiểm tra hai điều kiện bằng `rank`. Gặp `u` thoả thì đếm và `break`.

In [10]:
def unhappy_rank(n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
    rank = [[0] * n for _ in range(n)]
    for x in range(n):
        for i, u in enumerate(preferences[x]):
            rank[x][u] = i
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x

    ans = 0
    for x in range(n):
        y = partner[x]
        for u in range(n):
            if u == x or u == y:
                continue
            if rank[x][u] < rank[x][y] and rank[u][x] < rank[u][partner[u]]:
                ans += 1
                break
    return ans

for n, prefs, pairs, want in examples:
    print(unhappy_rank(n, prefs, pairs), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0
4 | mong đợi 4


Code nộp LeetCode cho cách 2:

```python
class Solution:
    def unhappyFriends(self, n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
        rank = [[0] * n for _ in range(n)]
        for x in range(n):
            for i, u in enumerate(preferences[x]):
                rank[x][u] = i
        partner = [0] * n
        for x, y in pairs:
            partner[x], partner[y] = y, x
        ans = 0
        for x in range(n):
            y = partner[x]
            for u in range(n):
                if u != x and u != y and rank[x][u] < rank[x][y] and rank[u][x] < rank[u][partner[u]]:
                    ans += 1
                    break
        return ans
```

**Độ phức tạp:** `O(n²)` thời gian, `O(n²)` bộ nhớ cho `rank`.

### 7.2 Cách 3 — Chỉ duyệt tiền tố "được thích hơn" + `any` (lời giải gọn)

Thay vì duyệt mọi `u`, chỉ duyệt `preferences[x][:rank[x][y]]` — những người `x` thích hơn `y`. Điều kiện 1 tự động đúng, chỉ cần kiểm tra điều kiện 2. Dùng `any` để dừng sớm.

In [11]:
def unhappy_prefix(n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
    rank = [[0] * n for _ in range(n)]
    for x in range(n):
        for i, u in enumerate(preferences[x]):
            rank[x][u] = i
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    return sum(
        any(rank[u][x] < rank[u][partner[u]] for u in preferences[x][:rank[x][partner[x]]])
        for x in range(n)
    )

for n, prefs, pairs, want in examples:
    print(unhappy_prefix(n, prefs, pairs), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0
4 | mong đợi 4


Code nộp LeetCode cho cách 3 (giống phần 9):

```python
class Solution:
    def unhappyFriends(self, n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
        rank = [[0] * n for _ in range(n)]
        for x in range(n):
            for i, u in enumerate(preferences[x]):
                rank[x][u] = i
        partner = [0] * n
        for x, y in pairs:
            partner[x], partner[y] = y, x
        return sum(
            any(rank[u][x] < rank[u][partner[u]] for u in preferences[x][:rank[x][partner[x]]])
            for x in range(n)
        )
```

**Độ phức tạp:** vẫn `O(n²)` trường hợp xấu nhất, nhưng thường nhanh hơn nhiều vì chỉ duyệt tiền tố và dừng sớm.

### 7.3 Cách 4 — Tập "thích hơn bạn cặp" + tìm cặp chặn

Với mỗi người `p`, đặt `better[p]` = **tập** những người `p` thích hơn bạn cặp của mình. Khi đó `(x, u)` là cặp chặn ⇔ `u ∈ better[x]` và `x ∈ better[u]`. Duyệt mọi `x`, mọi `u ∈ better[x]`; nếu `x ∈ better[u]` thì thêm **cả `x` và `u`** vào tập `unhappy` (nhờ tính đối xứng). Không cần ma trận `rank` đầy đủ, chỉ cần `index` của bạn cặp.

In [12]:
def unhappy_blocking_pairs(n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    better = [set(preferences[p][:preferences[p].index(partner[p])]) for p in range(n)]
    unhappy = set()
    for x in range(n):
        for u in better[x]:
            if x in better[u]:
                unhappy.add(x)
                unhappy.add(u)
    return len(unhappy)

for n, prefs, pairs, want in examples:
    print(unhappy_blocking_pairs(n, prefs, pairs), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0
4 | mong đợi 4


Code nộp LeetCode cho cách 4:

```python
class Solution:
    def unhappyFriends(self, n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
        partner = [0] * n
        for x, y in pairs:
            partner[x], partner[y] = y, x
        better = [set(preferences[p][:preferences[p].index(partner[p])]) for p in range(n)]
        unhappy = set()
        for x in range(n):
            for u in better[x]:
                if x in better[u]:
                    unhappy.add(x)
                    unhappy.add(u)
        return len(unhappy)
```

**Độ phức tạp:** `O(n²)` thời gian trung bình (tạo các tập tổng cộng `O(n²)`, mỗi phép `in` `O(1)` trung bình), `O(n²)` bộ nhớ trường hợp xấu.

## 8. TRACE từng bước

Cách 3 với ví dụ 3: `preferences = [[1,3,2],[2,3,0],[1,3,0],[0,2,1]]`, cặp `(1,3)`, `(0,2)` ⇒ `partner = [2, 3, 0, 1]`.

| x | y | rank[x][y] | tiền tố thích hơn | kiểm tra u | kết quả |
|---|---|---|---|---|---|
| 0 | 2 | 2 | `[1, 3]` | u=1: rank[1][0]=2 < rank[1][3]=1? ✘; u=3: rank[3][0]=0 < rank[3][1]=2? ✔ | không vui |
| 1 | 3 | 1 | `[2]` | u=2: rank[2][1]=0 < rank[2][0]=2? ✔ | không vui |
| 2 | 0 | 2 | `[1, 3]` | u=1: rank[1][2]=0 < rank[1][3]=1? ✔ | không vui |
| 3 | 1 | 2 | `[0, 2]` | u=0: rank[0][3]=1 < rank[0][2]=2? ✔ | không vui |

Kết quả `4`.

In [13]:
def trace_unhappy(n, preferences, pairs):
    rank = [[0] * n for _ in range(n)]
    for x in range(n):
        for i, u in enumerate(preferences[x]):
            rank[x][u] = i
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    print("partner =", partner)
    ans = 0
    for x in range(n):
        y = partner[x]
        prefix = preferences[x][:rank[x][y]]
        print(f"x={x}, y={y}, rank[x][y]={rank[x][y]}, thích hơn y: {prefix}")
        found = None
        for u in prefix:
            v = partner[u]
            ok = rank[u][x] < rank[u][v]
            print(f"    u={u} (ghép với v={v}): rank[u][x]={rank[u][x]} < rank[u][v]={rank[u][v]}? {ok}")
            if ok:
                found = u
                break
        if found is not None:
            ans += 1
        print(f"  -> {'KHÔNG VUI (bằng chứng u=' + str(found) + ')' if found is not None else 'vui'}; ans={ans}")
    return ans

trace_unhappy(4, [[1, 3, 2], [2, 3, 0], [1, 3, 0], [0, 2, 1]], [[1, 3], [0, 2]])

partner = [2, 3, 0, 1]
x=0, y=2, rank[x][y]=2, thích hơn y: [1, 3]
    u=1 (ghép với v=3): rank[u][x]=2 < rank[u][v]=1? False
    u=3 (ghép với v=1): rank[u][x]=0 < rank[u][v]=2? True
  -> KHÔNG VUI (bằng chứng u=3); ans=1
x=1, y=3, rank[x][y]=1, thích hơn y: [2]
    u=2 (ghép với v=0): rank[u][x]=0 < rank[u][v]=2? True
  -> KHÔNG VUI (bằng chứng u=2); ans=2
x=2, y=0, rank[x][y]=2, thích hơn y: [1, 3]
    u=1 (ghép với v=3): rank[u][x]=0 < rank[u][v]=1? True
  -> KHÔNG VUI (bằng chứng u=1); ans=3
x=3, y=1, rank[x][y]=2, thích hơn y: [0, 2]
    u=0 (ghép với v=2): rank[u][x]=1 < rank[u][v]=2? True
  -> KHÔNG VUI (bằng chứng u=0); ans=4


4

In [14]:
trace_unhappy(4, [[1, 2, 3], [3, 2, 0], [3, 1, 0], [1, 2, 0]], [[0, 1], [2, 3]])

partner = [1, 0, 3, 2]
x=0, y=1, rank[x][y]=0, thích hơn y: []
  -> vui; ans=0
x=1, y=0, rank[x][y]=2, thích hơn y: [3, 2]
    u=3 (ghép với v=2): rank[u][x]=0 < rank[u][v]=1? True
  -> KHÔNG VUI (bằng chứng u=3); ans=1
x=2, y=3, rank[x][y]=0, thích hơn y: []
  -> vui; ans=1
x=3, y=2, rank[x][y]=1, thích hơn y: [1]
    u=1 (ghép với v=0): rank[u][x]=0 < rank[u][v]=2? True
  -> KHÔNG VUI (bằng chứng u=1); ans=2


2

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def unhappyFriends(self, n: int, preferences: List[List[int]], pairs: List[List[int]]) -> int:
        rank = [[0] * n for _ in range(n)]
        for x in range(n):
            for i, u in enumerate(preferences[x]):
                rank[x][u] = i
        partner = [0] * n
        for x, y in pairs:
            partner[x], partner[y] = y, x
        return sum(
            any(rank[u][x] < rank[u][partner[u]] for u in preferences[x][:rank[x][partner[x]]])
            for x in range(n)
        )

sol = Solution()
for n, prefs, pairs, want in examples:
    print(sol.unhappyFriends(n, prefs, pairs), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0
4 | mong đợi 4


## 10. Vì sao thuật toán đúng

**Bảng `rank`.** Vì `preferences[x]` là một hoán vị của những người khác `x`, mỗi `u ≠ x` có đúng một vị trí `rank[x][u]`. "x thích u hơn y" ⇔ `u` đứng trước `y` ⇔ `rank[x][u] < rank[x][y]`. Vậy điều kiện kiểm tra bằng `rank` **tương đương nguyên văn** với định nghĩa.

**Cách 3 duyệt đủ.** Mọi `u` thoả điều kiện 1 đều nằm trong `preferences[x][:rank[x][y]]` và ngược lại; `u` trong tiền tố không thể là `x` (không có trong danh sách) hay `y` (đứng ở vị trí `rank[x][y]`, ngoài tiền tố). `any` trả `True` ⇔ tồn tại `u` thoả điều kiện 2 ⇔ `x` không vui. Mỗi `x` đóng góp đúng `0` hoặc `1` vào tổng ⇒ đếm **số người**, không phải số cặp.

**Cách 4 (đối xứng).** `x` không vui ⇔ ∃`u`: `u ∈ better[x]` ∧ `x ∈ better[u]`. Biểu thức này đối xứng theo `(x, u)`, nên khi tìm thấy cặp, cả hai đều không vui; dùng `set` tránh đếm trùng. Mọi `x` không vui đều được thêm khi duyệt chính `x` (với `u` là bằng chứng), nên không sót.

## 11. Phân tích Time / Space

Gọi `n` = số người.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force (index, dò pairs) | `O(n³)` | `O(1)` |
| 2. `rank` + `partner`, duyệt mọi u | `O(n²)` | `O(n²)` |
| 3. `rank` + tiền tố + `any` | `O(n²)` xấu nhất, thường nhỏ hơn nhiều | `O(n²)` |
| 4. Tập `better` + cặp chặn | `O(n²)` trung bình | `O(n²)` |

Xây bảng `rank` đã tốn `Θ(n²)` (đọc toàn bộ đầu vào kích thước `n(n-1)`), nên `O(n²)` là tối ưu theo kích thước input.

## 12. Bộ test

In [16]:
ALL = [brute_force, unhappy_rank, unhappy_prefix, unhappy_blocking_pairs, Solution().unhappyFriends]

for f in ALL:
    for n, prefs, pairs, want in examples:
        assert f(n, prefs, pairs) == want, (f.__name__, n)
print("Ví dụ đề: OK")

# Edge case tự dựng (đã kiểm chứng bằng brute force)
edge = [
    # mọi người ghép với người mình thích nhất -> 0
    (4, [[1, 2, 3], [0, 2, 3], [3, 0, 1], [2, 0, 1]], [[0, 1], [2, 3]], 0),
    # 0 và 2 thích nhau nhất nhưng bị ghép với người khác -> 0, 2 không vui
    (4, [[2, 1, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]], [[0, 1], [2, 3]], 2),
]
for f in ALL:
    for n, prefs, pairs, want in edge:
        assert f(n, prefs, pairs) == want, (f.__name__, prefs, f(n, prefs, pairs))
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
# Ngẫu nhiên seed cố định, đúng ràng buộc (n chẵn, preferences là hoán vị, mỗi người đúng một cặp)
rng = random.Random(1583)
for _ in range(1500):
    n = rng.choice([2, 4, 6, 8, 10])
    prefs, pairs = random_instance(n, rng)
    want = brute_force(n, prefs, pairs)
    for f in ALL[1:]:
        assert f(n, prefs, pairs) == want, (f.__name__, n, prefs, pairs)
print("1500 bộ ngẫu nhiên: OK")

# Tính chất đối xứng: tập người không vui luôn là hợp các cặp chặn -> mỗi người không vui có bằng chứng cũng không vui
def unhappy_set(n, prefs, pairs):
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    better = [set(prefs[p][:prefs[p].index(partner[p])]) for p in range(n)]
    return {x for x in range(n) if any(x in better[u] for u in better[x])}

for _ in range(300):
    n = rng.choice([4, 6, 8])
    prefs, pairs = random_instance(n, rng)
    S = unhappy_set(n, prefs, pairs)
    assert len(S) == brute_force(n, prefs, pairs)
print("Kiểm tra tập người không vui: OK")

1500 bộ ngẫu nhiên: OK
Kiểm tra tập người không vui: OK


In [18]:
# Test lớn: n = 500 (giới hạn đề)
rng = random.Random(500)
for trial in range(3):
    prefs, pairs = random_instance(500, rng)
    res = []
    for f in [unhappy_rank, unhappy_prefix, unhappy_blocking_pairs]:
        t0 = time.perf_counter()
        res.append(f(500, prefs, pairs))
        print(f"  {f.__name__:<24} {res[-1]:>4}  {time.perf_counter() - t0:.3f}s")
    assert len(set(res)) == 1

# Trường hợp đặc biệt n = 500: ai cũng ghép với người mình thích nhất -> 0
prefs = []
for i in range(500):
    mate = i ^ 1
    prefs.append([mate] + [j for j in range(500) if j != i and j != mate])
pairs = [[i, i + 1] for i in range(0, 500, 2)]
assert Solution().unhappyFriends(500, prefs, pairs) == 0
print("Test lớn: OK")

  unhappy_rank              496  0.017s


  unhappy_prefix            496  0.017s


  unhappy_blocking_pairs    496  0.024s


  unhappy_rank              500  0.015s


  unhappy_prefix            500  0.016s
  unhappy_blocking_pairs    500  0.024s


  unhappy_rank              498  0.016s


  unhappy_prefix            498  0.017s


  unhappy_blocking_pairs    498  0.023s
Test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm **giá trị** với **thứ hạng**: dùng `preferences[x][u]` như thể là vị trí của `u`

`preferences[x][u]` là người đứng ở vị trí `u` trong danh sách của `x`, **không phải** vị trí của người `u`.

In [19]:
def wrong_value_as_rank(n, preferences, pairs):
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    ans, index_errors = 0, 0
    for x in range(n):
        y = partner[x]
        for u in range(n):
            if u in (x, y):
                continue
            try:
                # SAI: preferences[x][u] là "người ở vị trí u", không phải "thứ hạng của u"
                cond = preferences[x][u] < preferences[x][y] and preferences[u][x] < preferences[u][partner[u]]
            except IndexError:
                index_errors += 1          # chỉ số u có thể = n-1, vượt độ dài n-1 của danh sách
                continue
            if cond:
                ans += 1
                break
    return ans, index_errors

for n, prefs, pairs, want in [examples[0], examples[2]]:
    got, errs = wrong_value_as_rank(n, prefs, pairs)
    print("sai:", got, f"(gặp {errs} lần IndexError bị bỏ qua)", "| đúng:", Solution().unhappyFriends(n, prefs, pairs))

sai: 0 (gặp 5 lần IndexError bị bỏ qua) | đúng: 2
sai: 0 (gặp 6 lần IndexError bị bỏ qua) | đúng: 4


### Lỗi 2 — Đếm **số cặp (x, u)** thay vì **số người**

Quên `break` ⇒ một người không vui có nhiều bằng chứng bị đếm nhiều lần.

In [20]:
def wrong_count_pairs(n, preferences, pairs):
    rank = [[0] * n for _ in range(n)]
    for x in range(n):
        for i, u in enumerate(preferences[x]):
            rank[x][u] = i
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    ans = 0
    for x in range(n):
        for u in preferences[x][:rank[x][partner[x]]]:
            if rank[u][x] < rank[u][partner[u]]:
                ans += 1                       # SAI: thiếu break
    return ans

n, prefs, pairs, want = examples[2]
print("sai:", wrong_count_pairs(n, prefs, pairs), "| đúng:", Solution().unhappyFriends(n, prefs, pairs))

sai: 6 | đúng: 4


### Lỗi 3 — Chỉ kiểm tra điều kiện của `x`, quên điều kiện của `u`

"`x` thích `u` hơn `y`" thôi chưa đủ — `u` cũng phải thích `x` hơn bạn cặp của `u`.

In [21]:
def wrong_one_condition(n, preferences, pairs):
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    # SAI: x không vui chỉ cần có người x thích hơn bạn cặp
    return sum(1 for x in range(n) if preferences[x].index(partner[x]) > 0)

n, prefs, pairs = 4, [[2, 1, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]], [[0, 1], [2, 3]]
print("sai:", wrong_one_condition(n, prefs, pairs), "| đúng:", Solution().unhappyFriends(n, prefs, pairs))

sai: 3 | đúng: 2


### Lỗi 4 — Tận dụng đối xứng nhưng cộng `+2` cho mỗi cặp chặn, không dùng `set`

Một người có thể thuộc nhiều cặp chặn ⇒ bị cộng nhiều lần; và mỗi cặp còn bị gặp hai lần (từ `x` và từ `u`).

In [22]:
def wrong_plus_two(n, preferences, pairs):
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    better = [set(preferences[p][:preferences[p].index(partner[p])]) for p in range(n)]
    ans = 0
    for x in range(n):
        for u in better[x]:
            if x in better[u]:
                ans += 2                      # SAI: đếm trùng
    return ans

n, prefs, pairs, want = examples[0]
print("sai:", wrong_plus_two(n, prefs, pairs), "| đúng:", Solution().unhappyFriends(n, prefs, pairs))

sai: 4 | đúng: 2


### Lỗi 5 — Xây `rank` ngược chiều: `rank[x][i] = preferences[x][i]`

Đó chỉ là sao chép lại danh sách, không phải đảo ngược. Phải là `rank[x][preferences[x][i]] = i`.

In [23]:
def wrong_rank_build(n, preferences, pairs):
    rank = [[0] * n for _ in range(n)]
    for x in range(n):
        for i, u in enumerate(preferences[x]):
            rank[x][i] = u                    # SAI: phải là rank[x][u] = i
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    return sum(
        any(rank[u][x] < rank[u][partner[u]] for u in range(n) if u not in (x, partner[x]) and rank[x][u] < rank[x][partner[x]])
        for x in range(n)
    )

n, prefs, pairs, want = examples[2]
print("sai:", wrong_rank_build(n, prefs, pairs), "| đúng:", Solution().unhappyFriends(n, prefs, pairs))

sai: 3 | đúng: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"So sánh mức độ yêu thích nhiều lần"** ⇒ dựng **bảng thứ hạng** (đảo ngược danh sách ưu tiên). Đây là bước chuẩn trong thuật toán Gale–Shapley.
- **"Tồn tại …"** ⇒ `any` / `break` sớm; **đếm số người** ⇒ mỗi người đóng góp 0/1.
- Định nghĩa dài ⇒ viết brute force nguyên văn trước, rồi tối ưu từng phần tra cứu.
- Chú ý tính đối xứng của cặp chặn — có thể giảm một nửa công việc hoặc dùng để kiểm tra chéo.
- Kiểm tra nhanh: ai cũng ghép với người thích nhất ⇒ `0`; `n = 2` ⇒ luôn `0`.

## 15. Biến thể của bài

1. **Liệt kê** những người không vui kèm một bằng chứng `u` (bài tập 3).
2. **Kiểm tra ghép cặp ổn định** (không có ai không vui) — đây là khái niệm *stable roommates*.
3. **Gale–Shapley (stable marriage):** hai phía (nam/nữ), tìm ghép cặp ổn định — luôn tồn tại (bài tập 5).
4. **Stable roommates (một phía, như bài này):** không phải lúc nào cũng tồn tại ghép cặp ổn định; thuật toán Irving giải trong `O(n²)`.
5. **Đếm cặp chặn** thay vì người không vui: duyệt `x < u` với điều kiện đối xứng.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | dịch nguyên văn, `index` + dò `pairs` | `O(n³)` | `O(1)` | kiểm thử, `n` nhỏ |
| 2. `rank` + `partner` | tra cứu `O(1)`, duyệt mọi `u` | `O(n²)` | `O(n²)` | lời giải chuẩn, dễ hiểu |
| 3. Tiền tố + `any` | chỉ duyệt người thích hơn bạn cặp, dừng sớm | `O(n²)` (thường nhỏ hơn) | `O(n²)` | nhanh nhất trong thực tế, code ngắn |
| 4. Tập `better` | cặp chặn `u ∈ better[x] ∧ x ∈ better[u]` | `O(n²)` | `O(n²)` | khi muốn liệt kê cặp chặn / tận dụng đối xứng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `n = 4`, `preferences = [[2,1,3],[0,2,3],[0,1,3],[0,1,2]]`, `pairs = [[0,1],[2,3]]`. Có bao nhiêu người không vui? Là những ai?

**Bài 2 (tính tay):** Với `preferences = [[1,3,2],[2,3,0],[1,3,0],[0,2,1]]`, viết bảng `rank` (4×4, bỏ qua đường chéo).

**Bài 3 (điền TODO):** Viết `ex3_unhappy_with_witness(n, preferences, pairs)` trả về `dict` `{x: u}` cho mỗi người không vui `x`, với `u` là bằng chứng **được `x` thích nhất**.

**Bài 4 (tự viết lại):** Viết `ex4_count_blocking_pairs(n, preferences, pairs)` đếm số **cặp chặn** `{x, u}` (không thứ tự).

**Bài 5 (mini challenge — Gale–Shapley):** Cho `k` nam và `k` nữ, mỗi người có danh sách ưu tiên toàn bộ phía bên kia. Viết `ex5_gale_shapley(men_prefs, women_prefs)` trả về `wife[m]` cho mỗi nam `m`, sao cho **không có cặp chặn** (nam `m` và nữ `w` thích nhau hơn bạn đời hiện tại).

In [24]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_unhappy_with_witness(n, preferences, pairs):
    res = {}
    # TODO: xây rank và partner
    # TODO: với mỗi x, duyệt preferences[x] theo thứ tự (người thích nhất trước) tới trước partner[x]
    # TODO: u đầu tiên có rank[u][x] < rank[u][partner[u]] là bằng chứng -> res[x] = u
    return res

In [25]:
# Bài 5 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex5_gale_shapley(men_prefs, women_prefs):
    k = len(men_prefs)
    # TODO: rank_w[w][m] = vị trí của m trong danh sách của w
    # TODO: hàng đợi các nam chưa ghép; next_idx[m] = nữ tiếp theo m sẽ cầu hôn
    # TODO: w chấp nhận m nếu đang tự do hoặc thích m hơn chồng hiện tại
    return [None] * k

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `partner = [1, 0, 3, 2]`. (2) Người 0 thích 2 nhất; người 2 thích 0 nhất. (3) `(0, 2)` là cặp chặn; kiểm tra thêm người 1 và 3.

**Bài 2.** (1) `rank[x][preferences[x][i]] = i`. (2) Hàng 0: người 1 ở vị trí 0, người 3 ở vị trí 1, người 2 ở vị trí 2. (3) Lặp cho từng hàng.

**Bài 3.** (1) Giống cách 3 nhưng giữ lại `u` đầu tiên. (2) Duyệt theo thứ tự danh sách nên `u` đầu tiên là người được thích nhất. (3) Số khoá của `dict` bằng đáp án.

**Bài 4.** (1) Dùng `better[p]`. (2) Chỉ đếm khi `x < u` để không trùng. (3) Kiểm tra bằng brute force trên mọi cặp.

**Bài 5.** (1) Mỗi vòng, một nam tự do cầu hôn nữ kế tiếp trong danh sách. (2) Nữ so sánh bằng `rank_w`. (3) Kiểm tra ổn định: không tồn tại `(m, w)` mà `m` thích `w` hơn vợ và `w` thích `m` hơn chồng.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
n1, p1, pr1 = 4, [[2, 1, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]], [[0, 1], [2, 3]]
print("Bài 1:", Solution().unhappyFriends(n1, p1, pr1), "người:", sorted(unhappy_set(n1, p1, pr1)))
assert Solution().unhappyFriends(n1, p1, pr1) == 2 and unhappy_set(n1, p1, pr1) == {0, 2}

# Bài 2
p2 = [[1, 3, 2], [2, 3, 0], [1, 3, 0], [0, 2, 1]]
rank2 = [[None] * 4 for _ in range(4)]
for x in range(4):
    for i, u in enumerate(p2[x]):
        rank2[x][u] = i
for row in rank2:
    print("   ", row)
assert rank2 == [[None, 0, 2, 1], [2, None, 0, 1], [2, 0, None, 1], [0, 2, 1, None]]
print("Bài 1, 2: OK")

Bài 1: 2 người: [0, 2]
    [None, 0, 2, 1]
    [2, None, 0, 1]
    [2, 0, None, 1]
    [0, 2, 1, None]
Bài 1, 2: OK


In [27]:
# Bài 3
def ex3_unhappy_with_witness_sol(n, preferences, pairs):
    rank = [[0] * n for _ in range(n)]
    for x in range(n):
        for i, u in enumerate(preferences[x]):
            rank[x][u] = i
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    res = {}
    for x in range(n):
        for u in preferences[x][:rank[x][partner[x]]]:
            if rank[u][x] < rank[u][partner[u]]:
                res[x] = u
                break
    return res

print("Bài 3:", ex3_unhappy_with_witness_sol(*examples[0][:3]), ex3_unhappy_with_witness_sol(*examples[2][:3]))
rng = random.Random(3)
for _ in range(500):
    n = rng.choice([2, 4, 6, 8])
    prefs, pairs = random_instance(n, rng)
    res = ex3_unhappy_with_witness_sol(n, prefs, pairs)
    assert len(res) == brute_force(n, prefs, pairs)
    partner = {a: b for a, b in pairs} | {b: a for a, b in pairs}
    for x, u in res.items():
        assert prefs[x].index(u) < prefs[x].index(partner[x]) and prefs[u].index(x) < prefs[u].index(partner[u])
print("Bài 3: OK")

Bài 3: {1: 3, 3: 1} {0: 3, 1: 2, 2: 1, 3: 0}
Bài 3: OK


In [28]:
# Bài 4
def ex4_count_blocking_pairs_sol(n, preferences, pairs):
    partner = [0] * n
    for x, y in pairs:
        partner[x], partner[y] = y, x
    better = [set(preferences[p][:preferences[p].index(partner[p])]) for p in range(n)]
    return sum(1 for x in range(n) for u in better[x] if x < u and x in better[u])

def ex4_brute(n, preferences, pairs):
    partner = {a: b for a, b in pairs} | {b: a for a, b in pairs}
    cnt = 0
    for x in range(n):
        for u in range(x + 1, n):
            if u == partner[x]:
                continue
            if preferences[x].index(u) < preferences[x].index(partner[x]) and preferences[u].index(x) < preferences[u].index(partner[u]):
                cnt += 1
    return cnt

print("Bài 4:", ex4_count_blocking_pairs_sol(*examples[0][:3]), ex4_count_blocking_pairs_sol(*examples[2][:3]))
rng = random.Random(4)
for _ in range(500):
    n = rng.choice([2, 4, 6, 8, 10])
    prefs, pairs = random_instance(n, rng)
    assert ex4_count_blocking_pairs_sol(n, prefs, pairs) == ex4_brute(n, prefs, pairs)
print("Bài 4: OK")

Bài 4: 1 3
Bài 4: OK


In [29]:
# Bài 5 — Gale–Shapley
from collections import deque

def ex5_gale_shapley_sol(men_prefs, women_prefs):
    k = len(men_prefs)
    rank_w = [[0] * k for _ in range(k)]
    for w in range(k):
        for i, m in enumerate(women_prefs[w]):
            rank_w[w][m] = i
    husband = [None] * k
    wife = [None] * k
    next_idx = [0] * k
    free = deque(range(k))
    while free:
        m = free.popleft()
        w = men_prefs[m][next_idx[m]]
        next_idx[m] += 1
        h = husband[w]
        if h is None:
            husband[w], wife[m] = m, w
        elif rank_w[w][m] < rank_w[w][h]:
            husband[w], wife[m], wife[h] = m, w, None
            free.append(h)
        else:
            free.append(m)
    return wife

def is_stable(wife, men_prefs, women_prefs):
    k = len(wife)
    husband = [None] * k
    for m, w in enumerate(wife):
        husband[w] = m
    for m in range(k):
        for w in men_prefs[m][:men_prefs[m].index(wife[m])]:
            if women_prefs[w].index(m) < women_prefs[w].index(husband[w]):
                return False
    return True

men = [[0, 1, 2], [1, 0, 2], [0, 1, 2]]
women = [[1, 0, 2], [0, 1, 2], [0, 1, 2]]
w5 = ex5_gale_shapley_sol(men, women)
print("Bài 5: wife =", w5, "| ổn định?", is_stable(w5, men, women))
assert sorted(w5) == [0, 1, 2] and is_stable(w5, men, women)
rng = random.Random(5)
for _ in range(400):
    k = rng.randint(1, 7)
    men = [rng.sample(range(k), k) for _ in range(k)]
    women = [rng.sample(range(k), k) for _ in range(k)]
    w = ex5_gale_shapley_sol(men, women)
    assert sorted(w) == list(range(k)) and is_stable(w, men, women)
print("Bài 5: OK")

Bài 5: wife = [0, 1, 2] | ổn định? True
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def unhappyFriends(self, n, preferences, pairs):
        rank = [[0] * n for _ in range(n)]
        for x in range(n):
            for i, u in enumerate(preferences[x]):
                rank[x][u] = i                       # đảo ngược danh sách ưu tiên
        partner = [0] * n
        for x, y in pairs:
            partner[x], partner[y] = y, x
        return sum(
            any(rank[u][x] < rank[u][partner[u]]     # u thích x hơn bạn cặp của u
                for u in preferences[x][:rank[x][partner[x]]])   # những người x thích hơn bạn cặp
            for x in range(n)
        )
```

**Template "so sánh ưu tiên":** `rank[x][preferences[x][i]] = i` → "x thích a hơn b" ⇔ `rank[x][a] < rank[x][b]`.

**Checklist trước khi submit:**
- [ ] `rank[x][u] = i` (không phải `rank[x][i] = u`).
- [ ] Kiểm tra **cả hai** điều kiện (của `x` và của `u`).
- [ ] Đếm **người**, mỗi `x` tối đa 1 lần (`break`/`any` hoặc `set`).
- [ ] `partner` gán hai chiều.
- [ ] Thử nhanh: ví dụ → `2, 0, 4`; ai cũng ghép người thích nhất → `0`.